# Campaign throughput

How many 230 h batches can Penicillium simulate per minute when a campaign runs in parallel? This notebook measures it
the way a user would run a campaign: `simulate_batches` on a list of `BatchSpec`, Raman spectra off, the Numba backend,
one worker process per logical CPU. The timings include starting the worker processes, so they are what a user waits
for, not just the time spent integrating.

The numbers depend on the machine; the first cell records it. Run the notebook again to measure yours.

In [1]:
import ctypes
import os
import platform
import statistics
import subprocess
import time

import numpy as np
import pandas as pd

import penicillium
from penicillium import BatchSpec, RamanMode, SimulationConfig, Simulator, simulate_batches


def cpu_name():
    try:
        if platform.system() == "Windows":
            import winreg

            key = winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r"HARDWARE\DESCRIPTION\System\CentralProcessor\0")
            return winreg.QueryValueEx(key, "ProcessorNameString")[0].strip()
        if platform.system() == "Darwin":
            return subprocess.check_output(["sysctl", "-n", "machdep.cpu.brand_string"], text=True).strip()
        with open("/proc/cpuinfo") as f:
            return next(line.split(":", 1)[1].strip() for line in f if line.startswith("model name"))
    except Exception:
        return platform.processor() or platform.machine()


def ram_gb():
    try:
        if platform.system() == "Windows":

            class MemoryStatus(ctypes.Structure):
                _fields_ = [("dwLength", ctypes.c_ulong), ("dwMemoryLoad", ctypes.c_ulong)] + [
                    (name, ctypes.c_ulonglong)
                    for name in ("total", "avail", "totalPage", "availPage", "totalVirtual", "availVirtual", "ext")
                ]

            status = MemoryStatus()
            status.dwLength = ctypes.sizeof(MemoryStatus)
            ctypes.windll.kernel32.GlobalMemoryStatusEx(ctypes.byref(status))
            return status.total / 2**30
        if platform.system() == "Darwin":
            return int(subprocess.check_output(["sysctl", "-n", "hw.memsize"], text=True)) / 2**30
        return os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
    except Exception:
        return float("nan")


CONFIG = SimulationConfig(raman_mode=RamanMode.OFF)
N_CPU = os.cpu_count()
print(f"Penicillium {penicillium.__version__}, Python {platform.python_version()}, {platform.system()}")
print(f"CPU: {cpu_name()}, {N_CPU} logical CPUs; RAM: {ram_gb():.0f} GB")

Penicillium 0.1.0.dev0, Python 3.14.4, Windows
CPU: Intel(R) Core(TM) i7-7700HQ CPU @ 2.80GHz, 8 logical CPUs; RAM: 16 GB


## One batch at a time

For reference, the time of a single default batch in this process (best and median of five seeds, after one warm-up
batch that loads the compiled model).

In [2]:
sim = Simulator(config=CONFIG)
sim.run(seed=0)  # warm-up: loads the compiled model from the cache
serial = []
for seed in range(1, 6):
    t = time.perf_counter()
    sim.run(seed=seed)
    serial.append(time.perf_counter() - t)
print(
    f"one batch: best {min(serial):.2f} s, median {statistics.median(serial):.2f} s "
    f"-> {60 / statistics.median(serial):.0f} batches per minute in series"
)

one batch: best 0.56 s, median 0.61 s -> 98 batches per minute in series


## A 200-batch campaign in parallel

200 different batches (RNG seeds 1-200) on one worker per logical CPU, timed three times. A short warm-up campaign
first makes sure every worker finds the compiled model in the on-disk cache. Each timed run starts a new pool of
worker processes, as a user's campaign does.

In [3]:
specs = [BatchSpec(seed=s, config=CONFIG) for s in range(1, 201)]
simulate_batches(specs[:N_CPU], keep_raman=False)  # warm-up of the workers' cache

runs = []
for rep in range(1, 4):
    t = time.perf_counter()
    results = simulate_batches(specs, keep_raman=False)
    dt = time.perf_counter() - t
    yields = np.array([r.total_yield for r in results])
    assert len(results) == len(specs) and np.all(np.isfinite(yields))  # every batch completed
    runs.append(
        {
            "run": rep,
            "batches": len(results),
            "seconds": round(dt, 1),
            "batches per minute": round(len(results) / dt * 60),
        }
    )
runs = pd.DataFrame(runs).set_index("run")
print(f"{len(np.unique(yields.round(3)))} distinct batches; mean yield {yields.mean():.0f} kg")
runs

200 distinct batches; mean yield 2929 kg


,batches,seconds,batches per minute
run,,,
1,200,33.7,356
2,200,35.8,336
3,200,33.7,356


In [4]:
per_min = statistics.median(runs["batches per minute"])
print(
    f"Median throughput: {per_min:.0f} batches per minute on {N_CPU} worker processes "
    f"({60 / per_min:.2f} s per batch on average)"
)

Median throughput: 356 batches per minute on 8 worker processes (0.17 s per batch on average)


## How throughput grows with the number of workers

The same 96 batches on 1, 2, 4 and all workers. With one worker the batches run in this process, one after the
other; with more, the time also includes starting the pool of worker processes, which takes a few seconds and weighs
more on short campaigns. On a CPU with hyper-threading, the logical CPUs beyond the physical cores share a core with
another worker and add less than a core of their own.

In [5]:
few = specs[:96]
scaling = []
for n in sorted({1, 2, 4, N_CPU}):
    t = time.perf_counter()
    simulate_batches(few, n_jobs=n, keep_raman=False)
    dt = time.perf_counter() - t
    scaling.append({"workers": n, "seconds": round(dt, 1), "batches per minute": round(len(few) / dt * 60)})
scaling = pd.DataFrame(scaling).set_index("workers")
scaling["speed-up"] = (scaling["batches per minute"] / scaling["batches per minute"].iloc[0]).round(1)
scaling

,seconds,batches per minute,speed-up
workers,,,
1,61.4,94,1.0
2,35.3,163,1.7
4,22.0,262,2.8
8,18.5,311,3.3
